# Fish Audio S2 Pro — Kaggle T4 ×2 Production Demo / Demo Production trên Kaggle T4 ×2

> **English:** Canonical public Kaggle notebook for the Fish Audio S2 Pro dual-T4 workflow.  
> **Tiếng Việt:** Notebook Kaggle public canonical cho quy trình Fish Audio S2 Pro trên hai GPU T4.

## Overview / Tổng quan

**English**

This notebook is designed for a fresh Kaggle session and keeps inference logic in the repository's canonical scripts. It separates **technical reference-free coverage** from the **reviewer-facing listening path**.

- Six reference-free English/Vietnamese FP16 cases remain the technical bilingual baseline.
- Reviewer-facing Vietnamese listening and local-API examples use the retained **independent VoxCPM2 Vietnamese reference bank** already qualified by this release.
- Human listening of both reference-free and same-language VoxCPM2-conditioned Vietnamese found only modest improvement; native Vietnamese pronunciation / tonal realization remains **NOT QUALIFIED**. This is treated as a tested model-capability boundary, not a runtime execution failure.

The showcase demonstrates exact **NVIDIA Tesla T4 ×2** gating, split model placement, six reference-free EN/VI technical cases, the 8-speaker / 12-case S2 Pro clone matrix using independent VoxCPM2 references, reviewer-facing audio playback, local HTTP API requests, timing/memory evidence, and a final machine-readable scorecard.

**Tiếng Việt**

Notebook này dành cho một Kaggle session mới và giữ inference logic trong các script canonical của repository. Notebook tách rõ **coverage kỹ thuật reference-free** khỏi **đường nghe thử dành cho reviewer**.

- Sáu case FP16 English/Vietnamese reference-free vẫn là baseline kỹ thuật song ngữ.
- Phần nghe thử và local API tiếng Việt dùng **ngân hàng giọng tham chiếu tiếng Việt độc lập do VoxCPM2 tạo** đã được release này qualification.
- Human listening với cả reference-free và same-language VoxCPM2-conditioned Vietnamese chỉ ghi nhận cải thiện nhỏ; native Vietnamese pronunciation / tonal realization vẫn **NOT QUALIFIED**. Đây được xem là giới hạn capability của model đã kiểm tra, không phải lỗi thực thi runtime.

Showcase chứng minh hard gate đúng **NVIDIA Tesla T4 ×2**, placement tách model rõ ràng, sáu case kỹ thuật EN/VI reference-free, ma trận clone S2 Pro 8 speaker / 12 case dùng reference VoxCPM2 độc lập, audio playback cho reviewer, request local HTTP API thật, timing/memory evidence và final machine-readable scorecard.

### Qualified topology / Kiến trúc đã qualification

| Component / Thành phần | Device |
|---|---|
| Semantic / Dual-AR | `cuda:0` — Tesla T4 |
| Codec / reference encoder / decoder | `cuda:1` — Tesla T4 |
| Precision | FP16 |
| Batch size | 1 |
| `max_seq_len` | 4096 |
| `torch.compile` | OFF |

**Built with Fish Audio.**


## Before you run / Trước khi chạy

### A. Import this notebook from GitHub / Import notebook từ GitHub

**English**
1. Create a **new Kaggle Notebook**.
2. Choose **File → Import Notebook → GitHub**.
3. Search for `dangkhoa2016/Fish-Audio-S2-Pro-On-Kaggle-T4x2-GPU`.
4. Select `notebooks/kaggle-production-demo.ipynb`.

**Tiếng Việt**
1. Tạo **Kaggle Notebook mới**.
2. Chọn **File → Import Notebook → GitHub**.
3. Tìm `dangkhoa2016/Fish-Audio-S2-Pro-On-Kaggle-T4x2-GPU`.
4. Chọn `notebooks/kaggle-production-demo.ipynb`.

### B. Configure Kaggle / Cấu hình Kaggle

**English**
1. Set **Accelerator → GPU T4 ×2**.
2. Turn **Internet ON** so the notebook can clone the public source and prepare the runtime.
3. In **Input**, choose **Add Input → Models**.
4. Attach **`dangkhoa2016/fishaudio-s2-pro`**.
5. Choose **PyTorch → default → version 1** when Kaggle asks for framework/variation/version.
6. Confirm the model appears at `/kaggle/input/models/dangkhoa2016/fishaudio-s2-pro/pytorch/default/1`.

The VoxCPM2 reference WAV files used by the listening showcase are already versioned in the repository under `results/synthetic-speaker-bank/`; no second model attachment is required.

**Tiếng Việt**
1. Chọn **Accelerator → GPU T4 ×2**.
2. Bật **Internet ON** để notebook clone source public và chuẩn bị runtime.
3. Trong **Input**, chọn **Add Input → Models**.
4. Attach **`dangkhoa2016/fishaudio-s2-pro`**.
5. Chọn **PyTorch → default → version 1** khi Kaggle yêu cầu framework/variation/version.
6. Xác nhận model xuất hiện tại `/kaggle/input/models/dangkhoa2016/fishaudio-s2-pro/pytorch/default/1`.

Các WAV tham chiếu VoxCPM2 dùng cho phần nghe thử đã được version trong repository tại `results/synthetic-speaker-bank/`; không cần attach thêm model thứ hai.

### C. Dataset requirement / Yêu cầu Dataset

**English:** No external Kaggle Dataset is required. Runtime source comes from GitHub, model weights come from the attached Kaggle Model, and the synthetic reference bank is retained in Git as release evidence.

**Tiếng Việt:** Không cần Kaggle Dataset ngoài. Runtime source đến từ GitHub, model weights đến từ Kaggle Model đã attach, còn synthetic reference bank được giữ trong Git như release evidence.

### D. Publication run / Cách chạy để publish

**English:** Use **Restart Session → Run All** exactly once on a fresh T4×2 session. Do not repair only selected cells and present that as a clean publication run.

**Tiếng Việt:** Dùng **Restart Session → Run All** đúng một lần trên session T4×2 mới. Không chạy sửa riêng từng cell rồi coi đó là một publication run sạch.

Expected final marker / Marker cuối cần thấy:

`KAGGLE_PRODUCTION_DEMO_TECHNICAL=PASS`

Language-quality boundary / Ranh giới chất lượng ngôn ngữ: `VIETNAMESE_NATIVE_PRONUNCIATION=NOT_QUALIFIED`


## Step 1/5 — Bootstrap + hardware + Kaggle Input / Bootstrap + phần cứng + Kaggle Input

### Goal / Mục tiêu

**English:** Prepare the exact public source, verify T4×2, validate the model attachment, checkout the pinned Fish Speech revision, apply the dual-GPU patch, and build the runtime.

**Tiếng Việt:** Chuẩn bị đúng public source, xác minh T4×2, kiểm tra model attachment, checkout Fish Speech revision đã pin, apply dual-GPU patch và dựng runtime.

### What the next code cell does / Cell code bên dưới làm gì

**English**
- clones a fresh checkout of public `v1.0.0`;
- prints exact repository HEAD and requires a clean tree;
- hard-fails unless the first two CUDA devices are T4;
- verifies the exact Kaggle model mount;
- runs `preflight.sh`, `bootstrap_kaggle.sh`, `apply_upstream_patch.sh`, and `setup_runtime.sh`;
- prints the model manifest and fixed split-device contract.

**Tiếng Việt**
- clone một checkout sạch từ public `v1.0.0`;
- in chính xác repository HEAD và yêu cầu worktree sạch;
- fail ngay nếu hai CUDA device đầu không phải T4;
- xác minh đúng Kaggle model mount;
- chạy `preflight.sh`, `bootstrap_kaggle.sh`, `apply_upstream_patch.sh`, và `setup_runtime.sh`;
- in model manifest và split-device contract cố định.

### Expected evidence / Kết quả cần thấy

- `PUBLIC_SOURCE=PASS`
- `GPU_T4X2=PASS`
- `KAGGLE_MODEL_INPUT=PASS`
- `UPSTREAM_PIN=PASS`
- `PATCH_APPLIED=PASS` or / hoặc `PATCH_ALREADY_APPLIED=PASS`
- `RUNTIME_SETUP=PASS`
- `NOTEBOOK_PHASE_SETUP=PASS`


In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, sys, time
import torch

REPO_URL = "https://github.com/dangkhoa2016/Fish-Audio-S2-Pro-On-Kaggle-T4x2-GPU.git"
SOURCE_REF = os.environ.get("FISH_AUDIO_SOURCE_REF", "v1.0.0")
WORK = Path("/kaggle/working")
ROOT = WORK / "fish-audio-s2-pro-public"
UPSTREAM = WORK / "fish-speech-upstream"
MODEL = Path("/kaggle/input/models/dangkhoa2016/fishaudio-s2-pro/pytorch/default/1")

def run(args, *, cwd=None, env=None):
    merged = os.environ.copy()
    if env:
        merged.update(env)
    print("+", " ".join(map(str, args)), flush=True)
    return subprocess.run(args, cwd=cwd, env=merged, check=True, text=True)

if ROOT.exists():
    shutil.rmtree(ROOT)
run(["git","clone","-q","--depth","1","--branch",SOURCE_REF,REPO_URL,str(ROOT)])
head = subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
status = subprocess.check_output(["git","status","--porcelain"],cwd=ROOT,text=True)
assert not status.strip(), status
print("PUBLIC_SOURCE=PASS")
print("PUBLIC_SOURCE_REF=", SOURCE_REF)
print("PUBLIC_SOURCE_HEAD=", head)

assert torch.cuda.is_available(), "CUDA unavailable: select Kaggle GPU T4 x2."
assert torch.cuda.device_count() >= 2, f"Need 2 CUDA devices, found {torch.cuda.device_count()}."
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert all("T4" in gpu_names[i] for i in (0,1)), gpu_names[:2]
print("GPU0=", gpu_names[0])
print("GPU1=", gpu_names[1])
print("GPU_T4X2=PASS")

assert MODEL.exists(), f"Missing attached Kaggle Model: {MODEL}"
required = ["codec.pth","config.json","tokenizer.json","model.safetensors.index.json"]
missing = [x for x in required if not (MODEL/x).exists()]
assert not missing, missing
print("KAGGLE_MODEL_INPUT=PASS")
print("MODEL_DIR=", MODEL)

env = {"UPSTREAM_DIR": str(UPSTREAM)}
run(["bash","scripts/preflight.sh"], cwd=ROOT, env=env)
run(["bash","scripts/bootstrap_kaggle.sh"], cwd=ROOT, env=env)
run(["bash","scripts/apply_upstream_patch.sh"], cwd=ROOT, env=env)
run(["bash","scripts/setup_runtime.sh"], cwd=ROOT, env=env)

manifest = json.loads((ROOT/"results/model-manifest.json").read_text())
print("MODEL_FILES=", len(manifest.get("files", [])))
print("SEMANTIC_DEVICE=cuda:0")
print("DECODER_DEVICE=cuda:1")
print("NOTEBOOK_PHASE_SETUP=PASS")


## Step 2/5 — Reference-free bilingual FP16 baseline / Baseline FP16 song ngữ reference-free

### Goal / Mục tiêu

**English:** Run the canonical six-case reference-free qualification with one model load: short, medium and expressive speech for English and Vietnamese. This is a **technical generation baseline**, not a claim that every reference-free Vietnamese sample has native pronunciation.

**Tiếng Việt:** Chạy qualification reference-free sáu case canonical bằng một lần load model: short, medium và expressive cho English và Vietnamese. Đây là **baseline sinh audio kỹ thuật**, không phải tuyên bố rằng mọi mẫu tiếng Việt reference-free đều có phát âm như người bản ngữ.

### What the next code cell does / Cell code bên dưới làm gì

**English:** The cell calls `scripts/qualify_text_synthesis.py` exactly as the repository qualification path does and renders case ID, language, duration, inference time, RTF, peak GPU memory and clipping.

**Tiếng Việt:** Cell gọi `scripts/qualify_text_synthesis.py` đúng như qualification path của repository và hiển thị case ID, ngôn ngữ, thời lượng, inference time, RTF, peak GPU memory và clipping.

### Interpretation / Cách đọc kết quả

**English:** `PASS` means the case completed, produced a structurally valid WAV and met the technical checks. It does **not** qualify native Vietnamese accent, pronunciation, or tonal naturalness.

**Tiếng Việt:** `PASS` nghĩa là case chạy xong, tạo WAV hợp lệ về cấu trúc và vượt qua các kiểm tra kỹ thuật. Nó **không** qualification native Vietnamese accent, pronunciation hoặc tonal naturalness.

### Expected evidence / Kết quả cần thấy

- six cases with `status=PASS` / sáu case có `status=PASS`;
- `BILINGUAL_CASES=6/6`;
- zero clipping / không clipping;
- `NOTEBOOK_PHASE_BILINGUAL=PASS`.


In [ ]:
from IPython.display import Audio, display
import pandas as pd

bilingual_dir = ROOT / "results/bilingual-text-synthesis"
run([
    str(UPSTREAM/".venv/bin/python"),
    "scripts/qualify_text_synthesis.py",
    "--checkpoint", str(MODEL),
    "--output-dir", str(bilingual_dir),
    "--semantic-device","cuda:0",
    "--decoder-device","cuda:1",
    "--max-seq-len","4096",
], cwd=ROOT, env={"UPSTREAM_DIR":str(UPSTREAM),"PYTHONPATH":str(UPSTREAM)})

tts = json.loads((bilingual_dir/"summary.json").read_text())
assert tts["status"] == "PASS" and tts["case_count"] == 6
rows = [{
    "case": c["id"],
    "lang": c["language"],
    "category": c["category"],
    "audio_s": round(c["duration_s"],2),
    "inference_s": round(c["total_inference_s"],2),
    "RTF": round(c["rtf"],3),
    "GPU0_peak_GiB": round(c["gpu0_peak_allocated_bytes"]/2**30,2),
    "GPU1_peak_GiB": round(c["gpu1_peak_allocated_bytes"]/2**30,2),
    "clipping": c["clipping_ratio_ge_0_999"],
} for c in tts["cases"]]
display(pd.DataFrame(rows))
assert all(c["clipping_ratio_ge_0_999"] == 0 for c in tts["cases"])
print("BILINGUAL_CASES=6/6")
print("NOTEBOOK_PHASE_BILINGUAL=PASS")


## Step 3/5 — Reference-conditioned listening showcase / Showcase nghe thử có reference conditioning

### Goal / Mục tiêu

**English:** Use the retained independent VoxCPM2 speaker bank to exercise S2 Pro speaker conditioning and provide reviewer-facing English/Vietnamese examples. Same-language Vietnamese references test whether better conditioning changes pronunciation; human review found only modest improvement, so this step is speaker-conditioning evidence, not native-pronunciation qualification.

**Tiếng Việt:** Dùng ngân hàng speaker VoxCPM2 độc lập đã được giữ lại để chạy speaker conditioning của S2 Pro và cung cấp ví dụ English/Vietnamese cho reviewer. Same-language Vietnamese references được dùng để kiểm tra liệu conditioning tốt hơn có cải thiện pronunciation hay không; human review chỉ thấy cải thiện nhỏ, vì vậy đây là speaker-conditioning evidence, không phải native-pronunciation qualification.

### What the next code cell does / Cell code bên dưới làm gì

**English**
- runs `scripts/qualify_synthetic_multispeaker.py`;
- consumes the eight retained VoxCPM2 references (4 EN + 4 VI);
- generates 8 same-language and 4 cross-language S2 Pro clone cases;
- displays the complete 12-case timing/RTF table;
- plays representative EN and VI **same-language** outputs for pronunciation-focused review.

**Tiếng Việt**
- chạy `scripts/qualify_synthetic_multispeaker.py`;
- dùng tám reference VoxCPM2 đã giữ lại (4 EN + 4 VI);
- sinh 8 case cùng ngôn ngữ và 4 case cross-language bằng S2 Pro;
- hiển thị bảng timing/RTF đủ 12 case;
- phát các output EN và VI **cùng ngôn ngữ** đại diện để review phát âm.

### Expected evidence / Kết quả cần thấy

- `MULTISPEAKER_CASES=12/12`
- `REFERENCE_CONDITIONED_LISTENING_REVIEW=READY`
- `VIETNAMESE_REFERENCE_CONDITIONING=PASS`
- `VIETNAMESE_NATIVE_PRONUNCIATION=NOT_QUALIFIED`
- `NOTEBOOK_PHASE_REFERENCE_CONDITIONING=PASS`


In [ ]:
multi_dir = ROOT / "results/synthetic-multispeaker-acceptance"
run([
    str(UPSTREAM/".venv/bin/python"),
    "scripts/qualify_synthetic_multispeaker.py",
    "--repo-root", str(ROOT),
    "--checkpoint", str(MODEL),
    "--speaker-bank", "results/synthetic-speaker-bank/summary.json",
    "--output-dir", "results/synthetic-multispeaker-acceptance",
    "--semantic-device", "cuda:0",
    "--decoder-device", "cuda:1",
    "--max-seq-len", "4096",
], cwd=ROOT, env={"UPSTREAM_DIR":str(UPSTREAM),"PYTHONPATH":str(UPSTREAM)})

multi = json.loads((multi_dir/"summary.json").read_text())
assert multi["status"] == "PASS" and multi["case_count"] == 12 and multi["passed"] == 12

display(pd.DataFrame([{
    "case": c["id"],
    "source": c["source_lang"].upper(),
    "target": c["target_lang"].upper(),
    "mode": c["mode"],
    "audio_s": round(c["output"]["duration_s"], 2),
    "inference_s": round(c["timing"]["total_s"], 2),
    "RTF": round(c["timing"]["rtf"], 3),
    "GPU0_peak_GiB": round(c["peak_allocated_bytes"]["gpu0"]/2**30, 2),
    "GPU1_peak_GiB": round(c["peak_allocated_bytes"]["gpu1"]/2**30, 2),
} for c in multi["cases"]]))

for cid in ["en-f01-same", "en-m01-same", "vi-f01-same", "vi-m01-same"]:
    c = next(x for x in multi["cases"] if x["id"] == cid)
    print("\n" + "="*88)
    print(cid, f"| {c['source_lang'].upper()} reference -> {c['target_lang'].upper()} target")
    print("TARGET TEXT:", c["target_text"])
    print("REFERENCE GENERATOR:", c["reference"]["generator"])
    print("REFERENCE DESCRIPTION:", c["reference"]["description"])
    display(Audio(filename=str(ROOT/c["output"]["path"])))

print("MULTISPEAKER_CASES=12/12")
print("REFERENCE_CONDITIONED_LISTENING_REVIEW=READY")
print("VIETNAMESE_REFERENCE_CONDITIONING=PASS")
print("VIETNAMESE_NATIVE_PRONUNCIATION=NOT_QUALIFIED")
print("NOTEBOOK_PHASE_REFERENCE_CONDITIONING=PASS")


## Step 4/5 — Cross-language voice cloning review / Review voice cloning cross-language

### Goal / Mục tiêu

**English:** Review the cross-language subset from the same 12-case run without loading the model a second time. Cross-language cloning is retained strictly as technical speaker-conditioning evidence. Neither cross-language nor same-language cloning is treated as a native Vietnamese pronunciation-quality PASS gate.

**Tiếng Việt:** Review tập cross-language từ chính run 12 case ở Step 3 mà không load model lần thứ hai. Cross-language cloning được giữ hoàn toàn như bằng chứng kỹ thuật về speaker conditioning. Cả cross-language lẫn same-language cloning đều không được coi là native Vietnamese pronunciation-quality PASS gate.

### What the next code cell does / Cell code bên dưới làm gì

**English:** It selects the four cross-language cases already generated, prints their metrics, and plays one EN→VI plus one VI→EN example so reviewers can inspect the behavior directly.

**Tiếng Việt:** Cell chọn bốn case cross-language đã được sinh, in metrics và phát một ví dụ EN→VI cùng một ví dụ VI→EN để reviewer kiểm tra trực tiếp.

### Expected evidence / Kết quả cần thấy

- four cross-language cases / bốn case cross-language;
- all four `PASS`;
- `CROSS_LANGUAGE_CASES=4/4`;
- `NOTEBOOK_PHASE_VOICE_CLONING=PASS`.


In [ ]:
cross = [c for c in multi["cases"] if c["mode"] == "cross"]
assert len(cross) == 4 and all(c["status"] == "PASS" for c in cross)

display(pd.DataFrame([{
    "case": c["id"],
    "reference": c["source_lang"].upper(),
    "target": c["target_lang"].upper(),
    "audio_s": round(c["output"]["duration_s"], 2),
    "inference_s": round(c["timing"]["total_s"], 2),
    "RTF": round(c["timing"]["rtf"], 3),
} for c in cross]))

for cid in ["en-f01-cross-vi", "vi-f01-cross-en"]:
    c = next(x for x in cross if x["id"] == cid)
    print("\n" + "="*88)
    print(cid, f"| reference={c['source_lang'].upper()} -> target={c['target_lang'].upper()}")
    print("TARGET TEXT:", c["target_text"])
    display(Audio(filename=str(ROOT/c["output"]["path"])))

print("CROSS_LANGUAGE_CASES=4/4")
print("NOTEBOOK_PHASE_VOICE_CLONING=PASS")


## Step 5/5 — Reference-conditioned local HTTP API + final acceptance / Local HTTP API có reference + nghiệm thu cuối

### Goal / Mục tiêu

**English:** Prove that the same split-device FP16 runtime serves real local HTTP requests. Both reviewer-facing API examples use retained independent references; the Vietnamese request uses a same-language VoxCPM2 synthetic reference.

**Tiếng Việt:** Chứng minh cùng runtime split-device FP16 phục vụ request HTTP local thật. Cả hai ví dụ API dành cho reviewer đều dùng reference độc lập đã giữ lại; request tiếng Việt dùng synthetic reference VoxCPM2 cùng ngôn ngữ.

### What the next code cell does / Cell code bên dưới làm gì

**English**
1. starts the canonical API server on `127.0.0.1:8080`;
2. waits for `GET /v1/health = 200`;
3. loads one retained EN and one retained VI VoxCPM2 reference plus their exact transcripts;
4. sends one real English and one real Vietnamese `POST /v1/tts` with inline reference conditioning;
5. saves and plays both returned WAV files;
6. terminates the server cleanly;
7. writes `results/notebook-production-demo/production-demo-summary.json`.

**Tiếng Việt**
1. chạy canonical API server tại `127.0.0.1:8080`;
2. chờ `GET /v1/health = 200`;
3. đọc một reference EN và một reference VI VoxCPM2 cùng transcript chính xác;
4. gửi một request English và một request Vietnamese `POST /v1/tts` có inline reference conditioning;
5. lưu và phát cả hai WAV trả về;
6. dừng server sạch;
7. ghi `results/notebook-production-demo/production-demo-summary.json`.

### Expected final evidence / Evidence cuối cần thấy

- `LOCAL_API_HEALTH=PASS`
- `LOCAL_API_EN_TTS=PASS`
- `LOCAL_API_VI_TTS=PASS`
- `BILINGUAL_CASES=6/6`
- `MULTISPEAKER_CASES=12/12`
- `CROSS_LANGUAGE_CASES=4/4`
- `VIETNAMESE_PIPELINE_COMPATIBILITY=PASS`
- `VIETNAMESE_NATIVE_PRONUNCIATION=NOT_QUALIFIED`
- `KAGGLE_PRODUCTION_DEMO_TECHNICAL=PASS`

**English:** This is a production-style Kaggle demo, not an SLA/HA/multi-tenant hosted service.  
**Tiếng Việt:** Đây là production-style Kaggle demo, không phải dịch vụ hosted có SLA/HA/multi-tenant.


In [ ]:
import base64
import urllib.request

out = ROOT / "results/notebook-production-demo"
out.mkdir(parents=True, exist_ok=True)

bank = json.loads((ROOT/"results/synthetic-speaker-bank/summary.json").read_text())
ref_ids = {"en":"en-f01", "vi":"vi-f01"}
refs = {lang: next(x for x in bank["speakers"] if x["id"] == sid) for lang,sid in ref_ids.items()}

api_log = out / "api-server.log"
log_fh = open(api_log, "w")
api_env = os.environ.copy()
api_env.update({"UPSTREAM_DIR":str(UPSTREAM),"MODEL_DIR":str(MODEL),"LISTEN":"127.0.0.1:8080"})
api = subprocess.Popen(["bash","scripts/run_api_server_dual_gpu.sh"], cwd=ROOT, env=api_env,
                       stdout=log_fh, stderr=subprocess.STDOUT, text=True)

try:
    health = False
    for _ in range(150):
        if api.poll() is not None:
            raise RuntimeError(f"API exited early; inspect {api_log}")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8080/v1/health", timeout=2) as r:
                health = r.status == 200
                if health:
                    print(r.read().decode("utf-8", errors="replace"))
                    break
        except Exception:
            time.sleep(2)
    assert health
    print("LOCAL_API_HEALTH=PASS")

    api_outputs = []
    api_texts = {
        "en":"This English sentence was generated through the local Fish Audio S2 Pro API using an independent synthetic reference voice.",
        "vi":"Câu tiếng Việt này được tạo qua local API của Fish Audio S2 Pro bằng một giọng tham chiếu tổng hợp độc lập.",
    }

    for lang, text in api_texts.items():
        sp = refs[lang]
        ref_path = ROOT/"results/synthetic-speaker-bank"/sp["wav"]
        ref_b64 = base64.b64encode(ref_path.read_bytes()).decode("ascii")
        payload = json.dumps({
            "text": text,
            "format": "wav",
            "streaming": False,
            "seed": 42,
            "max_new_tokens": 256,
            "references": [{"audio": ref_b64, "text": sp["transcript"]}],
        }).encode()
        req = urllib.request.Request("http://127.0.0.1:8080/v1/tts", data=payload,
                                     headers={"Content-Type":"application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=240) as r:
            assert r.status == 200
            wav = out / f"api-{lang}.wav"
            wav.write_bytes(r.read())
        assert wav.stat().st_size > 44
        api_outputs.append((lang,text,sp,wav))
        print(f"LOCAL_API_{lang.upper()}_TTS=PASS")
finally:
    api.terminate()
    try:
        api.wait(timeout=15)
    except subprocess.TimeoutExpired:
        api.kill()
    log_fh.close()

for lang,text,sp,wav in api_outputs:
    print("\n" + "="*88)
    print("API", lang.upper(), "TEXT:", text)
    print("REFERENCE:", sp["id"], "| generator=", sp["generator"])
    display(Audio(filename=str(wav)))

summary = {
    "GPU_T4X2":"PASS",
    "KAGGLE_MODEL_INPUT":"PASS",
    "BILINGUAL_CASES":"6/6",
    "MULTISPEAKER_CASES":"12/12",
    "CROSS_LANGUAGE_CASES":"4/4",
    "REFERENCE_CONDITIONED_LISTENING_REVIEW":"READY",
    "VIETNAMESE_REFERENCE_CONDITIONING":"PASS",
    "LOCAL_API_HEALTH":"PASS",
    "LOCAL_API_EN_TTS":"PASS",
    "LOCAL_API_VI_TTS":"PASS",
    "VIETNAMESE_PIPELINE_COMPATIBILITY":"PASS",
    "VIETNAMESE_NATIVE_PRONUNCIATION":"NOT_QUALIFIED",
    "KAGGLE_PRODUCTION_DEMO_TECHNICAL":"PASS",
}
(out/"production-demo-summary.json").write_text(json.dumps(summary,indent=2)+"\n")
display(pd.DataFrame([{"check":k,"result":v} for k,v in summary.items()]))
for k,v in summary.items():
    print(f"{k}={v}")


## Reproducibility, scope and licensing / Tái lập, phạm vi và license

**English**
- The default notebook path is the qualified **dual-T4 FP16** topology.
- The six-case reference-free EN/VI run is retained as a technical generation baseline; it is not a subjective native-accent score.
- Reviewer-facing Vietnamese listening and API examples use retained independent VoxCPM2 Vietnamese synthetic references, but human listening found only modest improvement and native Vietnamese pronunciation remains NOT QUALIFIED.
- The eight-speaker VoxCPM2 bank and 12-case S2 Pro clone matrix are release evidence already stored in the repository; the notebook regenerates S2 Pro clone outputs from those frozen references.
- Single-T4 INT8 and two independent INT8 instances remain separately documented and are not mixed into this listening-oriented notebook.
- Model weights are read from the attached Kaggle Model and are never committed to Git.
- Public tunnel exposure is outside this notebook.
- Synthetic-reference acceptance does not claim universal equivalence to arbitrary real-human voice evaluation.
- Fish Audio S2 Pro, Fish Speech, and Fish Audio-derived patch material remain governed by the Fish Audio Research License.

**Tiếng Việt**
- Default path của notebook là topology **dual-T4 FP16** đã qualification.
- Run EN/VI reference-free sáu case được giữ làm baseline sinh audio kỹ thuật; không phải điểm số chủ quan về accent bản ngữ.
- Các ví dụ nghe thử và API tiếng Việt dành cho reviewer dùng synthetic reference tiếng Việt VoxCPM2 độc lập đã giữ lại, nhưng human listening chỉ ghi nhận cải thiện nhỏ và native Vietnamese pronunciation vẫn NOT QUALIFIED.
- Ngân hàng VoxCPM2 tám speaker và ma trận 12 case clone S2 Pro là release evidence đã có trong repository; notebook sinh lại output S2 Pro từ các reference đã đóng băng đó.
- Single-T4 INT8 và hai instance INT8 độc lập được tài liệu hóa riêng, không trộn vào notebook tập trung vào nghe thử.
- Model weights chỉ đọc từ Kaggle Model đã attach và không commit vào Git.
- Public tunnel nằm ngoài phạm vi notebook.
- Synthetic-reference acceptance không tuyên bố tương đương phổ quát với mọi giọng người thật.
- Fish Audio S2 Pro, Fish Speech và patch có nguồn gốc Fish Audio vẫn chịu Fish Audio Research License.

Xem / See `LICENSE-NOTES.md`, `NOTICE.txt`, and / và `THIRD_PARTY_LICENSES/FISH-AUDIO-RESEARCH-LICENSE`.

**Built with Fish Audio.**
